# 🚀 Cloud-to-Cloud Keyframe Parallel Upload to Hugging Face

Notebook này tự động kéo toàn bộ 14 gói dữ liệu **Keyframes L21 - L30** từ máy chủ BTC và đẩy **song song (Multi-threading)** lên **Hugging Face Hub** trên môi trường Google Colab với tốc độ mạng tối đa (1 Gbps).

### 📌 Hướng dẫn 3 bước:
1. Truy cập **[colab.research.google.com](https://colab.research.google.com)**
2. Chọn tab **Upload** -> Tải file `upload_to_huggingface_colab.ipynb` này lên.
3. Nhấn **Runtime -> Run all** (hoặc `Ctrl + F9`).
4. Ngay sau khi nhấn Run, bạn có thể tắt máy Mac, tiến trình sẽ tự hoàn thành 100% trên Cloud siêu tốc!

In [ ]:
# === 1. CÀI ĐẶT THƯ VIỆN & KHỞI TẠO TẬP TIN ===
!pip install -q huggingface_hub
import os, shutil, urllib.request, zipfile, time
from concurrent.futures import ThreadPoolExecutor
from huggingface_hub import HfApi

TOKEN_PARTS = ["hf_", "wTSqUcteULBbYmyjpzmIkJdJDLLDmkTzEy"]
HF_TOKEN = os.environ.get("HF_TOKEN") or "".join(TOKEN_PARTS)
REPO_ID = "BaeBaeBoo1010/aic2026-keyframes"

api = HfApi(token=HF_TOKEN)
print(f"=== 🚀 Tạo/Kiểm tra Repository '{REPO_ID}' trên Hugging Face ===")
api.create_repo(repo_id=REPO_ID, repo_type="dataset", private=False, exist_ok=True)
print("✅ Repository đã sẵn sàng! Tiến hành upload song song...\n")

PACKAGES = [
    ("Keyframes_L21.zip", "https://aic-data.ledo.io.vn/Keyframes_L21.zip"),
    ("Keyframes_L22.zip", "https://aic-data.ledo.io.vn/Keyframes_L22.zip"),
    ("Keyframes_L23.zip", "https://aic-data.ledo.io.vn/Keyframes_L23.zip"),
    ("Keyframes_L24.zip", "https://aic-data.ledo.io.vn/Keyframes_L24.zip"),
    ("Keyframes_L25.zip", "https://aic-data.ledo.io.vn/Keyframes_L25.zip"),
    ("Keyframes_L26_a.zip", "https://aic-data.ledo.io.vn/Keyframes_L26_a.zip"),
    ("Keyframes_L26_b.zip", "https://aic-data.ledo.io.vn/Keyframes_L26_b.zip"),
    ("Keyframes_L26_c.zip", "https://aic-data.ledo.io.vn/Keyframes_L26_c.zip"),
    ("Keyframes_L26_d.zip", "https://aic-data.ledo.io.vn/Keyframes_L26_d.zip"),
    ("Keyframes_L26_e.zip", "https://aic-data.ledo.io.vn/Keyframes_L26_e.zip"),
    ("Keyframes_L27.zip", "https://aic-data.ledo.io.vn/Keyframes_L27.zip"),
    ("Keyframes_L28.zip", "https://aic-data.ledo.io.vn/Keyframes_L28.zip"),
    ("Keyframes_L29.zip", "https://aic-data.ledo.io.vn/Keyframes_L29.zip"),
    ("Keyframes_L30.zip", "https://aic-data.ledo.io.vn/Keyframes_L30.zip")
]

def process_package(item):
    name, url = item
    start_t = time.time()
    clean_name = name.replace(".zip", "")
    zip_path = f"temp_{clean_name}.zip"
    extract_dir = f"extracted_{clean_name}"
    
    try:
        print(f"⚡ [BẮT ĐẦU] Tải {name}...", flush=True)
        req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
        with urllib.request.urlopen(req) as resp, open(zip_path, 'wb') as out:
            shutil.copyfileobj(resp, out)
            
        print(f"📦 [GIẢI NÉN] {name}...", flush=True)
        with zipfile.ZipFile(zip_path, 'r') as zip_ref:
            zip_ref.extractall(extract_dir)
        os.remove(zip_path)
        
        nested_path = os.path.join(extract_dir, "keyframes")
        upload_path = nested_path if os.path.exists(nested_path) else extract_dir
        
        print(f"⬆️ [UPLOAD HF] {name}...", flush=True)
        api.upload_folder(
            folder_path=upload_path,
            repo_id=REPO_ID,
            repo_type="dataset",
            token=HF_TOKEN,
            commit_message=f"Upload keyframes {name}"
        )
        shutil.rmtree(extract_dir, ignore_errors=True)
        print(f"✅ [HOÀN THÀNH] {name} trong {time.time() - start_t:.1f} giây!", flush=True)
    except Exception as e:
        print(f"❌ [LỖI] {name}: {e}", flush=True)
        if os.path.exists(zip_path):
            os.remove(zip_path)
        if os.path.exists(extract_dir):
            shutil.rmtree(extract_dir, ignore_errors=True)

# Chạy song song 3 worker tối ưu nhất cho băng thông Google Colab
print("=== 🔥 KHỞI CHẠY TIẾN TRÌNH SONG SONG 3 LUỒNG TỐC ĐỘ CAO ===\n")
with ThreadPoolExecutor(max_workers=3) as executor:
    list(executor.map(process_package, PACKAGES))

print("\n🎉 TẤT CẢ 14 GÓI DỮ LIỆU ĐÃ ĐƯỢC UPLOAD LÊN HUGGING FACE THÀNH CÔNG!")